# ETCCDI Climate Extremes Indices

Template notebook for computing ETCCDI climate indices.

## Imports

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

## Paths

In [ ]:
BASE = Path.cwd().parents[1]
DATA_DIR = BASE/'data'
OUTPUT_DIR = BASE/'output'/'ETCCDI'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load Data

In [ ]:
# ds_tx = xr.open_dataset('tasmax.nc')
# ds_tn = xr.open_dataset('tasmin.nc')
# ds_pr = xr.open_dataset('pr.nc')

## Temperature Indices

In [ ]:
txx = ds_tx['tasmax'].groupby('time.year').max(dim='time')
txn = ds_tx['tasmax'].groupby('time.year').min(dim='time')
tnx = ds_tn['tasmin'].groupby('time.year').max(dim='time')
tnn = ds_tn['tasmin'].groupby('time.year').min(dim='time')

## Precipitation Indices

In [ ]:
rx1day = ds_pr['pr'].groupby('time.year').max(dim='time')
rolling5 = ds_pr['pr'].rolling(time=5, min_periods=5).sum()
rx5day = rolling5.groupby('time.year').max(dim='time')

In [ ]:
wet_days = ds_pr['pr'].where(ds_pr['pr'] >= 1.0)
prcptot = wet_days.groupby('time.year').sum(dim='time')

In [ ]:
def longest_run(arr):
    max_run = run = 0
    for v in arr:
        if v:
            run += 1
            max_run = max(max_run, run)
        else:
            run = 0
    return max_run

In [ ]:
dry = ds_pr['pr'] < 1.0
cdd = dry.groupby('time.year').map(
    lambda x: xr.apply_ufunc(
        longest_run, x,
        input_core_dims=[['time']],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[int]
    )
)

In [ ]:
wet = ds_pr['pr'] >= 1.0
cwd = wet.groupby('time.year').map(
    lambda x: xr.apply_ufunc(
        longest_run, x,
        input_core_dims=[['time']],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[int]
    )
)

## Save Outputs

In [ ]:
txx.to_netcdf(OUTPUT_DIR/'TXx.nc')
txn.to_netcdf(OUTPUT_DIR/'TXn.nc')
tnx.to_netcdf(OUTPUT_DIR/'TNx.nc')
tnn.to_netcdf(OUTPUT_DIR/'TNn.nc')
rx1day.to_netcdf(OUTPUT_DIR/'RX1day.nc')
rx5day.to_netcdf(OUTPUT_DIR/'RX5day.nc')
prcptot.to_netcdf(OUTPUT_DIR/'PRCPTOT.nc')
cdd.to_netcdf(OUTPUT_DIR/'CDD.nc')
cwd.to_netcdf(OUTPUT_DIR/'CWD.nc')